# v4

In [490]:
%load_ext autoreload
%autoreload 2 

from src.route_cluster_pipeline_v4 import EnhancedH3RouteOptimizer
from src.get_data import get_processed_data
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [491]:
## 00. Load Data

# Load your data (replace with actual file paths)
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()  
df_output_assignment = pd.read_feather('./output/customer_assignments.feather')
# Preprocessing assignment
df_output_assignment['stock_point_id'] = df_output_assignment['stock_point_id'].astype(int)
df_output_assignment.dropna(subset=['cluster_id'], inplace=True) # Dropping customers without cluster assignment


⚠️ Warning: Dropped 806 customer records with coordinates outside Nigeria.
✅ Customer data cleaning complete. 28209 of 29015 records retained.


In [514]:
pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = int(pilot_sps_lists[1]) 

In [536]:
# Single stock point optimization
optimizer = EnhancedH3RouteOptimizer(
    sp_dim_df=sp_dim_df,
    customers_gdf=customers_gdf,
    df_output_assignment=df_output_assignment,
    stock_point_id=stock_point_id,
    min_customers=40,
    max_customers=100,
    max_distance_km=7
) 

# Run complete optimization pipeline
routes_df, validation_results = optimizer.optimize()

# Generate enhanced visualizations
map_viz = optimizer.create_enhanced_visualization(
    save_path=f'enhanced_route_map_{stock_point_id}.html',
    show_customer_points=True
)

# Export all results
# exported_files = optimizer.export_results()

#=f'route_map_{stock_point_id}.html', 

🚀 STARTING H3 ROUTE OPTIMIZATION PIPELINE
Phase 1: Enhanced Data Preparation & Validation
--------------------------------------------------
✓ Stock Point: OmniHub Apapa Lagos - CAUSEWAY (ID: 1647113)
✓ Fulfillment Center: (np.float64(6.473953), np.float64(3.356525))
✓ Unique H3 cells: 105
✓ Total customers: 1554
✓ Avg customers per cell: 14.8
✓ Cluster ID matches H3 Cell ID: True

Phase 2: Enhanced H3 Cell Metrics Calculation
--------------------------------------------------
✓ H3 cells within 7km: 96 (filtered out: 9)
✓ Total customers in valid cells: 1517
✓ Density range: 1.66 - 204.02 customers/km²

Phase 3: Advanced Geographic Clustering
--------------------------------------------------
  Distance calculation failed: Unknown Distance Metric: haversine, using simplified approach
  Checking for NaN values in features...
  Found NaN values: {'avg_neighbor_distance': np.int64(8)}
  ✓ NaN values handled
✓ Total customers: 1517
✓ Estimated routes needed: 21
  Testing K-means clustering

# v3

In [1]:
%load_ext autoreload
%autoreload 2 

from src.route_cluster_pipeline_v3 import H3RouteOptimizer
from src.get_data import get_processed_data
import pandas as pd

In [2]:
## 00. Load Data

# Load your data (replace with actual file paths)
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()  
df_output_assignment = pd.read_feather('./output/customer_assignments.feather')
# Preprocessing assignment
df_output_assignment['stock_point_id'] = df_output_assignment['stock_point_id'].astype(int)
df_output_assignment.dropna(subset=['cluster_id'], inplace=True) # Dropping customers without cluster assignment


⚠️ Warning: Dropped 806 customer records with coordinates outside Nigeria.
✅ Customer data cleaning complete. 28209 of 29015 records retained.


In [5]:
pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = int(pilot_sps_lists[1]) 

In [8]:
# Run optimization
optimizer = H3RouteOptimizer(sp_dim_df, customers_gdf, df_output_assignment, stock_point_id, 
                             min_customers=40, max_customers=100, max_distance_km=7)
routes_df, validation_results = optimizer.optimize()

# Generate route summary
route_summary_df = optimizer.generate_route_summary()

# Create visualization
map_viz = optimizer.create_route_visualization(
    save_path=f'./output/presentation/v3/route_map_{stock_point_id}_v3.html',
    show_customer_points=True
)

Starting H3 Route Optimization Pipeline
Phase 1: Data Preparation & Validation
Stock Point: OmniHub Apapa Lagos - CAUSEWAY (ID: 1647113)
Fulfillment Center: (np.float64(6.473953), np.float64(3.356525))
Unique H3 cells: 105
Total customers: 1554
Cluster ID matches H3 Cell ID: True
Phase 2: Calculate H3 Cell Metrics
H3 cells within 7km: 96
Total customers in valid cells: 1517
Phase 3: Improved Geographic Clustering
Estimated routes needed: 21
Using K-means clustering (score: 0.43)
Phase 4: Constraint Enforcement
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Convergence reached
Phase 5: Generate Output DataFrame
Phase 6: Validation & Summary

Validation Results:
Total routes: 18
Customer constraint violations: 1
Distance constraint violations: 0
Average customers per route: 84.3
Average distance per route: 3.4 km
Average compactness score: 0.524

Optimization Complete!
Generating Route Summary
Creating Route Visualization
Map saved to: ./output/presentation/v3/route_map_1647

In [ ]:
# # Initialize optimizer for specific stock point
# optimizer = H3RouteOptimizer(
#     sp_dim_df=sp_dim_df,
#     customers_gdf=customers_gdf, 
#     df_output_assignment=df_output_assignment,
#     stock_point_id=stock_point_id,  # Required parameter
#     min_customers=40,
#     max_customers=200,
#     max_distance_km=7
# )

# # Run optimization
# routes_df, validation_results = optimizer.optimize()

In [ ]:
routes_df.columns

# KeyError: "Column(s) ['max_vertex_distance_km', 'population_density']
# ['stock_point_id', 'stock_point_name', 'route_id', 'h3_cell_id',
# 'customer_count', 'total_distance_km', 'estimated_delivery_time_hours',
# 'compactness_score']

In [ ]:
route_summary.customer_count.sum()

In [ ]:
route_summary = (routes_df 
                .groupby('route_id')
                .agg(
                    h3_cell_ids = ('h3_cell_id', lambda x: x.to_list()),
                    cluster_count = ('h3_cell_id', 'nunique'), 
                    customer_count = ('customer_count', 'max'),  
                    total_distance_km = ('total_distance_km', 'max'),  
                    cumulative_distance_km = ('total_distance_km', 'max'),  
                    farthest_centroid_distance_km = ('total_distance_km', 'max'),  
                    estimated_delivery_time_hours = ('estimated_delivery_time_hours', 'max'),   
                    avg_assignment_confidence = ('estimated_delivery_time_hours', 'mean'),   
                    compactness_score = ('compactness_score', 'mean')   
                )
                .reset_index()
                )
    
# cols_to_cat = ['route_id']
# route_summary[cols_to_cat] = route_summary[cols_to_cat].astype('category')

In [ ]:
route_summary.columns

[]'stock_point_id', 'stock_point_name', 'route_id', 'h3_cell_ids', 'cluster_count', 'customer_count',
'total_distance_km', 'cumulative_distance_km',
'farthest_centroid_distance_km', 'estimated_delivery_time_hours',
'avg_assignment_confidence', 'compactness_score']

In [ ]:
# create_route_summary_barplot(route_summary, width=500, height=350)
m = create_route_map(route_summary, fc_coordinates=sp_coords, col_indx=9)
m

# v2

In [ ]:
# !pip install haversine

In [37]:
%load_ext autoreload
%autoreload 2 

# from src.route_cluster_pipeline_claudeTG import *
import pandas as pd
# from src.route_cluster_pipeline import main_route_planning_pipeline, calculate_optimization_metrics
from src.route_cluster_pipeline_v2 import *
from src.get_data import get_processed_data
from src.plot_utils import create_route_map, create_route_summary_barplot
from pathlib import Path 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
## 00. Load Data

# Load your data (replace with actual file paths)
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()  
df_output_assignment = pd.read_feather('./output/customer_assignments.feather')
# Preprocessing assignment
df_output_assignment['stock_point_id'] = df_output_assignment['stock_point_id'].astype(int)
df_output_assignment.dropna(subset=['cluster_id'], inplace=True) # Dropping customers without cluster assignment


⚠️ Warning: Dropped 806 customer records with coordinates outside Nigeria.
✅ Customer data cleaning complete. 28209 of 29015 records retained.


In [39]:
# Specify your target fulfillment center
pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = int(pilot_sps_lists[1]) 

# w1=1.0  # Population density weight
# w2=1.0  # Distance penalty weight
# distance_method='vertex'  # Conservative approach
# score_method='vertex'
# min_customers=40
# min_distance=3
# max_distance=7
# ROUTE_OUTPUT_PATH = Path('./output/routes') 

#### Notes 

The dbscan clustering of optimize_route_clustering is performing very poorly with 0 split  

The route split and merging might not gurantee geographically compactness  and max customer per route constraint


In [40]:
MAX_R2SP_DISTANCE_KM = 7
MIN_CUSTOMER_PER_ROUTE = 40
MAX_CUSTOMER_PER_ROUTE = 200

In [41]:
def main_optimization_workflow(stock_point_id):
    """Complete end-to-end optimization workflow"""

    # Phase 1: Data Preparation 
    validate_input_data(sp_dim_df, customers_gdf, df_output_assignment)

    # Extract fulfillment center coordinates
    fulfillment_center = extract_fulfillment_center(sp_dim_df, stock_point_id)
    sp_coords = fulfillment_center['coordinates']

    sp_assignments = df_output_assignment[
            df_output_assignment['stock_point_id'] == stock_point_id
        ]
    if sp_assignments.empty:
        raise ValueError(f"No assignments found for stock_point_id: {stock_point_id}")
        
    # # Phase 2: Calculate Metrics
    cluster_metrics_df = calculate_cluster_metrics(sp_assignments, sp_coords)
    valid_clusters_df = apply_distance_constraints(cluster_metrics_df, 
                                                max_distance_km=MAX_R2SP_DISTANCE_KM, 
                                                adjust_distance_threshold=False)
    if valid_clusters_df.empty:
        print('empty valid cluster for stockpoint: ', stock_point_id, " - ",fulfillment_center['stock_point_name'])
        cols = ['route_id', 'compactness_score', 'estimated_delivery_time_hours', 'total_distance_km']
        valid_clusters_df[cols] = None
        return {
            'cluster_metrics_df': cluster_metrics_df,
            'route_output_df': valid_clusters_df,
            'validation_report': {},
            'performance_metrics': {},
            'route_summary': pd.DataFrame(),
            'visualization_map': None
        }
    else:
        # Phase 3: Route Optimization
        cluster_labels, clustering_method = optimize_route_clustering(valid_clusters_df,
                                                                    min_customers=MIN_CUSTOMER_PER_ROUTE, 
                                                                    max_customers=MAX_CUSTOMER_PER_ROUTE)
        valid_clusters_df['initial_route_id'] = cluster_labels
        optimized_routes = enforce_route_constraints(valid_clusters_df,
                                                        min_customers=MIN_CUSTOMER_PER_ROUTE, 
                                                        max_customers=MAX_CUSTOMER_PER_ROUTE)

        # Phase 4: Calculate Statistics
        route_stats = calculate_route_statistics(optimized_routes, sp_coords)

        # Phase 5: Generate Output
        route_output_df_ = generate_output_dataframe(route_stats)
        route_output_df = route_output_df_.merge(cluster_metrics_df.drop(columns='customer_count'), on='h3_cell_id', how='inner')


        # # # Phase 6: Validation
        validation_report = validate_final_output(route_output_df)
        route_visualization = create_route_visualization(route_output_df, cluster_metrics_df, fulfillment_center)
        route_summary, performance_metrics = analyze_optimization_performance(route_output_df)
            
        
        print(performance_metrics) 
        return {
            'cluster_metrics_df': cluster_metrics_df,
            'route_output_df': route_output_df,
            'validation_report': validation_report,
            'performance_metrics': performance_metrics,
            'route_summary': route_summary,
            'visualization_map': route_visualization
        }

In [42]:
res_ = main_optimization_workflow(stock_point_id)

✓ Data validation passed
{'total_routes': 11, 'avg_customers_per_route': np.float64(139.63636363636363), 'avg_distance_per_route': np.float64(14.57805373553222), 'avg_delivery_time_hours': np.float64(2.614451343388305), 'avg_compactness_score': np.float64(0.5337678808494509), 'total_customers': np.int64(1536), 'total_clusters': np.int64(99)}


In [43]:
res_['visualization_map']

In [16]:
res_.keys() # ['route_output_df']

dict_keys(['cluster_metrics_df', 'route_output_df', 'validation_report', 'performance_metrics', 'route_summary', 'visualization_map'])

In [105]:
all_runs = []
for sp in pilot_sps_lists:
    run_ = main_optimization_workflow(int(sp))
    run_['route_output_df']['stock_point_id'] = int(sp)
    run_['cluster_metrics_df']['stock_point_id'] = int(sp)
    all_runs.append(run_)
    
    try:
        if run_['visualization_map']:
            run_['visualization_map'].save(f'./output/presentation/route_map_{sp}.html')
    except:
        print('cannot save file: ', sp)
    
interested_cols = all_runs[1]['route_output_df'].columns

df_all_route_assignment = []
df_all_cluster_metrics = []
for l in all_runs:
    df_ = l['route_output_df']
    dfcm = l['cluster_metrics_df']
    df_all_route_assignment.append(df_[interested_cols])
    df_all_cluster_metrics.append(dfcm)

df_all_routes = pd.concat(df_all_route_assignment)    
df_all_cluster_metrics = pd.concat(df_all_cluster_metrics).rename(columns={'h3_cell_id':'cluster_id',
                                                                           'centroid_distance_km':'centroid_distance_to_fc_km',
                                                                           'max_vertex_distance_km':'max_vertex_distance_to_fc_km'})  

✓ Data validation passed
empty valid cluster for stockpoint:  1647024  -  OmniHub Oyigbo Rivers - LAMDA GLOBAL
✓ Data validation passed
{'total_routes': 11, 'avg_customers_per_route': np.float64(139.63636363636363), 'avg_distance_per_route': np.float64(14.57805373553222), 'avg_delivery_time_hours': np.float64(2.614451343388305), 'avg_compactness_score': np.float64(0.5337678808494509), 'total_customers': np.int64(1536), 'total_clusters': np.int64(99)}
✓ Data validation passed
{'total_routes': 4, 'avg_customers_per_route': np.float64(100.0), 'avg_distance_per_route': np.float64(29.413443228481796), 'avg_delivery_time_hours': np.float64(5.360336080712045), 'avg_compactness_score': np.float64(0.32887826587640073), 'total_customers': np.int64(400), 'total_clusters': np.int64(74)}


In [49]:
df_all_routes.head(1)#.stock_point_id.unique()#head(1)

,route_id,h3_cell_id,customer_count,total_distance_km,estimated_delivery_time_hours,compactness_score,avg_confidence,dominant_tier,centroid_lat,centroid_lng,centroid_distance_km,max_vertex_distance_km,population_density,stock_point_id
0,0,88589c9941fffff,77,11.045502,2.776138,0.519503,1.0,h3_inclusion,6.488151,3.348487,1.811434,2.308359,19,1647113


In [18]:
# df_all_assignment.columns

In [19]:
# df_all_routes.columns

In [109]:
# All assignment
df_all_assignment = pd.read_feather('./output/customer_assignments.feather').drop(columns='h3_cell_id')
df_all_assignment['stock_point_id'] = df_all_assignment['stock_point_id'].astype(int)

df_all_routes = pd.concat(df_all_route_assignment).rename(columns={'h3_cell_id':'cluster_id'})
df_all_routes['stock_point_id'] = df_all_routes['stock_point_id'].astype(int)

df_unq_route = df_all_routes[['route_id', 'cluster_id','stock_point_id']].drop_duplicates()
df_assignment_and_clustering = df_all_assignment[df_all_assignment['stock_point_id'].isin([int(sp) for sp in pilot_sps_lists])].merge(df_unq_route, on=['cluster_id','stock_point_id'], how='left')


cm_cols = ['stock_point_id', 'cluster_id', 'customer_count', 'avg_confidence', 
 'dominant_tier', 'centroid_lat', 'centroid_lng', 'centroid_distance_to_fc_km', 
 'max_vertex_distance_to_fc_km']

df_all_cluster_metrics = df_all_cluster_metrics[cm_cols]


In [112]:
# Use ExcelWriter with a context manager
with pd.ExcelWriter(f'./output/presentation/pilot_assignment_and_route.xlsx', engine='openpyxl') as writer:
    # Write each DataFrame to a different sheet
    df_assignment_and_clustering.to_excel(writer, sheet_name='Assignment And Clustering', index=False)
    df_all_cluster_metrics.to_excel(writer, sheet_name='Cluster Metrics', index=False)  

In [17]:
res_['visualization_map']

In [ ]:
def main_optimization_workflow():
    """Complete end-to-end optimization workflow"""
    
    # Phase 1: Data Preparation
    # fc_coordinates, valid_h3_cells, _ = validate_datasets(sp_dim_df, customers_gdf, df_output_assignment)
    validate_datasets(sp_dim_df, customers_gdf, df_output_assignment)
    
    # # Phase 2: Calculate Metrics
    # cluster_metrics_df = calculate_cluster_metrics(df_output_assignment, fc_coordinates)
    # valid_clusters_df = apply_distance_constraints(cluster_metrics_df)
    
    # # Phase 3: Route Optimization
    # cluster_labels, clustering_method = optimize_route_clustering(valid_clusters_df)
    # valid_clusters_df['initial_route_id'] = cluster_labels
    # optimized_routes = enforce_route_constraints(valid_clusters_df)
    
    # # Phase 4: Calculate Statistics
    # route_stats = calculate_route_statistics(optimized_routes, fc_coordinates)
    
    # # Phase 5: Generate Output
    # final_output_df = generate_output_dataframe(route_stats)
    
    # # Phase 6: Validation
    # validation_report = validate_final_output(final_output_df)
    # route_visualization = create_route_visualization(final_output_df, cluster_metrics_df, fc_coordinates)
    # route_summary, performance_metrics = analyze_optimization_performance(final_output_df)
    
    # return {
    #     'output_dataframe': final_output_df,
    #     'validation_report': validation_report,
    #     'performance_metrics': performance_metrics,
    #     'route_summary': route_summary,
    #     'visualization_map': route_visualization
    # }

In [ ]:
# main_optimization_workflow() 

----

In [11]:
res_['route_summary'].head(2)
# cluster_metrics_df.head(2)
# final_output_df.head(2)
# final_output_df.merge(cluster_metrics_df.drop(columns='customer_count'), on='h3_cell_id')

,route_id,h3_cell_ids,cluster_count,customer_count,total_distance_km,cumulative_distance_km,farthest_centroid_distance_km,estimated_delivery_time_hours,avg_assignment_confidence,compactness_score
0,0,"[88589c9941fffff, 88589c9943fffff, 88589c9947f...",10,77,11.045502,11.045502,3.42609,2.776138,2.776138,0.519503
1,1,"[88589c9b01fffff, 88589c9b03fffff, 88589c9b05f...",8,86,17.111134,17.111134,7.23235,2.427778,2.427778,0.553210


In [13]:
  
create_route_summary_barplot(res_['route_summary'], width=500, height=350)


alt.Chart(...)

In [ ]:
# route_summary
performance_metrics

In [ ]:
route_visualization

In [15]:
m = create_route_map(res_['route_summary'], fc_coordinates=sp_coords, col_indx=9)
m

NameError: name 'sp_coords' is not defined

-----

# Implemenation 1 - Adjacent Joining

In [ ]:
%load_ext autoreload
%autoreload 2 

# from src.route_cluster_pipeline_claudeTG import *
import pandas as pd
# from src.route_cluster_pipeline import main_route_planning_pipeline, calculate_optimization_metrics
from src.route_cluster_pipeline import *
from src.get_data import get_processed_data
from src.plot_utils import create_route_map, create_route_summary_barplot
from pathlib import Path 

### Data

In [ ]:
## 00. Load Data

# Load your data (replace with actual file paths)
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()  
df_output_assignment = pd.read_feather('./output/customer_assignments.feather')
# Preprocessing assignment
df_output_assignment['stock_point_id'] = df_output_assignment['stock_point_id'].astype(int)
df_output_assignment.dropna(subset=['cluster_id'], inplace=True) # Dropping customers without cluster assignment


### Parameter

In [ ]:
# Specify your target fulfillment center
pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = int(pilot_sps_lists[2]) 

w1=1.0  # Population density weight
w2=1.0  # Distance penalty weight
distance_method='vertex'  # Conservative approach
score_method='vertex'
min_customers=40
min_distance=3
max_distance=7
ROUTE_OUTPUT_PATH = Path('./output/routes') 

In [ ]:
# optimized_output_df, optimized_routes, df_cluster_with_route = main_example(sp_dim_df=sp_dim_df, customers_gdf=customers_gdf, df_output_assignment=df_output_assignment, stock_point_id=stock_point_id)

## Run

In [ ]:
# print("\n🔬 Optimizing parameters...")
# results_df, best_config = optimize_parameters(
#     sp_dim_df, customers_gdf, df_output_assignment, stock_point_id
# ) 
# # Run with optimized parameters
# print("\n🏆 Running with optimized parameters...")
# optimized_output_df, optimized_routes, h3_dataset = main_route_planning_pipeline(
#     sp_dim_df=sp_dim_df,
#     customers_gdf=customers_gdf,
#     df_output_assignment=df_output_assignment,
#     stock_point_id=stock_point_id,
#     w1=best_config['w1'],
#     w2=best_config['w2'],
#     distance_method=best_config['distance_method'],
#     score_method=best_config['distance_method']
# )
        

In [ ]:
try:
    optimized_output_df, optimized_routes, h3_dataset = main_route_planning_pipeline(
        sp_dim_df, 
        customers_gdf, 
        df_output_assignment, 
        stock_point_id, 
        w1=1.0, 
        w2=1.0, 
        min_customers=40, 
        min_distance=0, 
        max_distance=7, 
        distance_method='vertex',  #'centroid', 
        score_method='vertex',  #'centroid',
        max_clusters_per_route=8, 
        max_customers_per_route=200,
        compactness_weight=0.3
    )


    print(calculate_optimization_metrics(output_df=optimized_output_df, optimization_metric='efficiency'))

    ## Post Processing 
    # Explode the 'h3_cell_ids' column
    df_route_cells_long = optimized_output_df[['route_id', 'h3_cell_ids']].explode('h3_cell_ids').rename(columns={'h3_cell_ids':'h3_cell_id'})

    df_cluster_with_route = h3_dataset.drop(columns='neighbors').merge(df_route_cells_long, on='h3_cell_id', how='left')
    df_cluster_with_route.isna().sum()
except Exception as e:
    print(f'Error: ',e)

In [ ]:
route_quality_df, quality_summary = validate_route_quality(optimized_output_df)
# quality_summary
print(route_quality_df.quality_issues.value_counts())
quality_summary

In [ ]:
calculate_optimization_metrics(optimized_output_df, optimization_metric='compactness')

In [ ]:
# Validate results
# validate_route_quality(optimized_output_df)
# generate_route_summary_statistics(optimized_output_df)

______

In [ ]:
create_route_summary_barplot(optimized_output_df, width=500, height=350)


----------------  

In [ ]:
m = create_route_map(optimized_output_df)
m

---

In [ ]:
# # Run with optimized parameters
# print("\n🏆 Running with optimized parameters...")
# optimized_output_df, optimized_routes, _ = main_route_planning_pipeline(
#     sp_dim_df=sp_dim_df,
#     customers_gdf=customers_gdf,
#     df_output_assignment=df_output_assignment,
#     stock_point_id=stock_point_id,
#     w1=best_config['w1'],
#     w2=best_config['w2'],
#     distance_method=best_config['distance_method'],
#     score_method=best_config['distance_method']
# )

In [ ]:
# # Save optimized results
# optimized_output_df.to_csv(ROUTE_OUTPUT_PATH / f'delivery_routes_sp_{stock_point_id}_optimized.csv', index=False)
# results_df.to_csv(ROUTE_OUTPUT_PATH / f'parameter_optimization_results_sp_{stock_point_id}.csv', index=False)

--------------------

#### Example Template
